In [2]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

file_path = "../sample.pdf"
loader = PyPDFLoader(file_path)
pages = loader.load()

document_content = "\n\n".join(doc.page_content for doc in pages)

In [6]:
llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.1,
        api_key=os.environ["GEMINI_API_KEY"]
    )

mindmap_prompt_template = """
당신은 복잡한 텍스트 정보를 분석하여 핵심 계층 구조를 파악하고, 이를 시각적인 마인드맵으로 정리하는 최고의 전문가입니다.
아래 제공된 [문서 내용]을 바탕으로, 사용자가 전체 내용을 한눈에 파악할 수 있도록 마인드맵을 생성해주세요.

**[출력 규칙]**
1. 가장 중요한 대주제를 # (h1)으로 표현합니다.
2. 주요 개념(가지)을 ## (h2)로 표현합니다.
3. 세부 항목들은 ### (h3) 또는 리스트(-)를 사용하여 들여쓰기로 표현합니다.
4. 각 항목은 핵심 키워드 중심으로 간결하게 요약해주세요.
5. 반드시 '들여쓰기를 사용한 마크다운(Markdown) 형식'으로만 출력해야 합니다.

---
[문서 내용]:
{document_content}
"""

mindmap_prompt = PromptTemplate.from_template(mindmap_prompt_template)
mindmap_chain = mindmap_prompt | llm | StrOutputParser()

mindmap_markdown = mindmap_chain.invoke({"document_content": document_content})

print(mindmap_markdown)

E0000 00:00:1759061694.532572   52716 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


# Attention Is All You Need (Transformer)

## 1. 개요 및 핵심 아이디어
### 1.1. 기존 모델의 한계
-   **지배적인 모델**: 복잡한 RNN 또는 CNN 기반 인코더-디코더.
-   **최고 성능 모델**: 어텐션 메커니즘을 통해 인코더-디코더 연결.
-   **RNN의 문제점**:
    -   본질적으로 순차적 계산: 병렬화 어려움.
    -   긴 시퀀스 길이에서 메모리 제약.
    -   장거리 의존성 학습의 어려움.

### 1.2. Transformer 제안
-   **새로운 아키텍처**: Transformer.
-   **핵심**: 오직 어텐션 메커니즘에만 기반.
-   **특징**: 순환(Recurrence) 및 합성곱(Convolution) 완전히 제거.

### 1.3. 주요 장점
-   **우수한 품질**: 기계 번역에서 SOTA 달성.
-   **높은 병렬화**: 훈련 시간 대폭 단축.
-   **훈련 시간 감소**: 기존 모델 대비 현저히 적은 시간 소요.
-   **일반화 능력**: 다른 태스크(예: 구문 분석)에도 성공적으로 적용.

## 2. 모델 아키텍처
### 2.1. 전체 구조
-   **인코더-디코더 구조**: 기존 경쟁력 있는 신경망 시퀀스 변환 모델과 동일.
    -   **인코더**: 입력 시퀀스를 연속적인 표현으로 매핑.
    -   **디코더**: 인코더 출력을 바탕으로 출력 시퀀스를 한 번에 하나씩 생성 (자기회귀적).
-   **Transformer 구성**: 스택형 셀프-어텐션 및 포인트-와이즈 완전 연결 레이어.

### 2.2. 인코더 및 디코더 스택
-   **인코더**: N=6개의 동일한 레이어 스택.
    -   **각 레이어**: 두 개의 서브-레이어.
        -   멀티-헤드 셀프-어텐션 메커니즘.
        -   포인트-와이즈 완전 연결 피드-포워드 네트워크.
    -   **연결**: 각 서브-레이어 주변에 잔차 연결(Res